In [ ]:
from pyspark.sql import functions as F

from config import gold_table, silver_table

In [ ]:
business_df = spark.table(silver_table("business"))
user_df = spark.table(silver_table("user"))
review_df = spark.table(silver_table("review"))
tip_df = spark.table(silver_table("tip"))

In [ ]:
dim_business = business_df.select(
    "business_id", "name", "city", "state", "categories", "is_open", "rating_band"
)

dim_business.write.format("delta").mode("overwrite").saveAsTable(gold_table("dim_business"))
print(f"dim_business: {dim_business.count()} linhas gravadas em {gold_table('dim_business')}")

In [ ]:
# Pivot: quantidade de business abertos/fechados por estado
business_by_state = (
    business_df
    .groupBy("state")
    .pivot("is_open", [0, 1])
    .count()
    .withColumnRenamed("0", "closed_count")
    .withColumnRenamed("1", "open_count")
    .na.fill(0, subset=["closed_count", "open_count"])
)

business_by_state.write.format("delta").mode("overwrite").saveAsTable(gold_table("business_by_state"))
print(f"business_by_state: {business_by_state.count()} linhas gravadas em {gold_table('business_by_state')}")

In [ ]:
dim_user = user_df.select("user_id", "name", "yelping_since")

dim_user.write.format("delta").mode("overwrite").saveAsTable(gold_table("dim_user"))
print(f"dim_user: {dim_user.count()} linhas gravadas em {gold_table('dim_user')}")

In [ ]:
# Media de notas (stars) e contagem de reviews por business
fact_business_metrics = review_df.groupBy("business_id").agg(
    F.count("*").alias("review_count"),
    F.avg("stars").alias("avg_stars"),
)

fact_business_metrics.write.format("delta").mode("overwrite").saveAsTable(gold_table("fact_business_metrics"))
print(f"fact_business_metrics: {fact_business_metrics.count()} linhas gravadas em {gold_table('fact_business_metrics')}")

In [ ]:
# Contagem de reviews e de tips por usuario
review_counts = review_df.groupBy("user_id").agg(F.count("*").alias("review_count"))
tip_counts = tip_df.groupBy("user_id").agg(F.count("*").alias("tip_count"))

fact_user_metrics = (
    review_counts.join(tip_counts, on="user_id", how="outer")
    .fillna(0, subset=["review_count", "tip_count"])
)

# Catalyst: plano fisico do join

fact_user_metrics.explain(mode="formatted")

fact_user_metrics.write.format("delta").mode("overwrite").saveAsTable(gold_table("fact_user_metrics"))
print(f"fact_user_metrics: {fact_user_metrics.count()} linhas gravadas em {gold_table('fact_user_metrics')}")

In [ ]:
# Pivot: quantidade de reviews por mes, uma linha por ano
MONTHS = ["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"]

review_by_month = (
    review_df
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
    .groupBy("year")
    .pivot("month", list(range(1, 13)))
    .count()
)

for month_number, month_name in enumerate(MONTHS, start=1):
    review_by_month = review_by_month.withColumnRenamed(str(month_number), month_name)

review_by_month = review_by_month.na.fill(0, subset=MONTHS)

review_by_month.write.format("delta").mode("overwrite").saveAsTable(gold_table("review_by_month"))
print(f"review_by_month: {review_by_month.count()} linhas gravadas em {gold_table('review_by_month')}")

In [ ]:
# KPIs globais: total de reviews, usuarios, businesses e a media de tips por usuario
total_reviews = review_df.count()
total_users = user_df.count()
total_businesses = business_df.count()
total_tips = tip_df.count()
avg_tips_per_user = total_tips / total_users if total_users else 0.0

summary_metrics = spark.createDataFrame(
    [(total_reviews, total_users, total_businesses, avg_tips_per_user)],
    "total_reviews BIGINT, total_users BIGINT, total_businesses BIGINT, avg_tips_per_user DOUBLE",
).withColumn("computed_at", F.current_timestamp())

summary_metrics.write.format("delta").mode("overwrite").saveAsTable(gold_table("summary_metrics"))
print(f"summary_metrics: {summary_metrics.count()} linha gravada em {gold_table('summary_metrics')}")